# Chat companion prompt eval

Runs the Phase 1 tagging prompt (`backend.chat.TAG_SYSTEM_PROMPT` / `TAG_SCHEMA`) against the labeled cases in `eval/chat_eval_cases.json`, plus a few prose-reply tone checks. Outputs are saved in this notebook on run, so results can be reviewed later without re-calling the API.

In [1]:
import json
import os
import sys
from pathlib import Path

# Make cwd the project root regardless of where this notebook is launched from,
# since st.secrets resolves .streamlit/secrets.toml relative to cwd.
if not (Path.cwd() / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from backend.chat import CHAT_MODEL, SYSTEM_PROMPT, TAG_SCHEMA, TAG_SYSTEM_PROMPT
from backend.claude_client import TAG_MODEL, call_prose, call_structured

CASES_PATH = Path.cwd() / "eval" / "chat_eval_cases.json"
cases = json.loads(CASES_PATH.read_text())
len(cases)

12

In [2]:
results = []
for i, case in enumerate(cases, start=1):
    messages = [*case["context"], {"role": "user", "content": case["message"]}]
    tags = call_structured(
        model=TAG_MODEL,
        system=TAG_SYSTEM_PROMPT,
        messages=messages,
        tool_name="tag_message",
        tool_description="Classify the sentiment and repetition of the latest message.",
        tool_schema=TAG_SCHEMA,
    )
    ok = (
        tags["sentiment"] == case["expected_sentiment"]
        and tags["repeated_question_flag"] == case["expected_repeated_question_flag"]
    )
    results.append(
        {
            "case": i,
            "message": case["message"],
            "expected_sentiment": case["expected_sentiment"],
            "got_sentiment": tags["sentiment"],
            "expected_repeated": case["expected_repeated_question_flag"],
            "got_repeated": tags["repeated_question_flag"],
            "pass": ok,
        }
    )

passed = sum(r["pass"] for r in results)
print(f"{passed}/{len(results)} passed\n")
for r in results:
    status = "PASS" if r["pass"] else "FAIL"
    print(f"[{status}] case {r['case']}: {r['message']!r}")
    print(
        f"         sentiment: got={r['got_sentiment']!r} expected={r['expected_sentiment']!r} | "
        f"repeated: got={r['got_repeated']!r} expected={r['expected_repeated']!r}"
    )

2026-07-29 20:55:32.520 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


12/12 passed

[PASS] case 1: 'I feel wonderful today, the weather is lovely!'
         sentiment: got='positive' expected='positive' | repeated: got=False expected=False
[PASS] case 2: 'Just checking in, nothing much happening today.'
         sentiment: got='neutral' expected='neutral' | repeated: got=False expected=False
[PASS] case 3: "I'm a bit tired today, didn't sleep well."
         sentiment: got='low' expected='low' | repeated: got=False expected=False
[PASS] case 4: "I don't see the point anymore, I feel so alone and hopeless."
         sentiment: got='distress' expected='distress' | repeated: got=False expected=False
[PASS] case 5: 'Did I take my medication today?'
         sentiment: got='neutral' expected='neutral' | repeated: got=True expected=True
[PASS] case 6: "What day is my doctor's appointment?"
         sentiment: got='neutral' expected='neutral' | repeated: got=False expected=False
[PASS] case 7: "I'm scared, someone called saying I owe money and I don't know what

## Prose reply tone check

Not a pass/fail eval — just eyeballing what the companion actually sounds like for a couple of representative messages, since tone matters as much as correctness here.

In [3]:
tone_check_messages = [
    "I've been feeling a bit lonely lately, nobody visits much anymore.",
    "Someone called saying my bank account will be frozen unless I pay them today.",
    "What's the weather like where you are?",
]

for msg in tone_check_messages:
    reply = call_prose(
        model=CHAT_MODEL, system=SYSTEM_PROMPT, messages=[{"role": "user", "content": msg}]
    )
    print(f"> {msg}\n{reply}\n{'-' * 60}")

> I've been feeling a bit lonely lately, nobody visits much anymore.
I'm really sorry to hear that, it can be so hard when days feel quiet. You're not alone right now, I'm here with you. 💛

Would it help to talk about what's been on your mind? Or maybe there's someone—a family member or old friend—you'd like to reach out to today, even just for a quick chat?
------------------------------------------------------------


> Someone called saying my bank account will be frozen unless I pay them today.
That sounds scary, but please don't pay anything yet — this has the signs of a scam. Real banks don't demand urgent payment by phone like that.

A few gentle tips:
- Don't give them money, card details, or codes.
- Hang up if you feel pressured.
- Call your bank directly using the number on your card or statement — not the one they gave you.

You could also use Point & Ask if you have anything in writing about this, or check with a family member before doing anything. Would you like help figuring out who to call?
------------------------------------------------------------


> What's the weather like where you are?
I don't have a real place I'm sitting in, so I don't feel the weather myself! But I'd love to hear about yours - is it sunny, rainy, or something in between where you are today?
------------------------------------------------------------
